# 🎯 XGBoost Model Training — Vertex AI Pipeline

This notebook packages and submits the **`ss-models` XGBoost model training job** to Vertex AI Custom Training.

### What this notebook does
1. **Loads** configuration from `config.yaml`
2. **Builds** the Python package (`setup.py sdist`)
3. **Uploads** the package to GCS
4. **Defines** a KFP pipeline with one custom training step
5. **Compiles** the pipeline to JSON
6. **Submits** the pipeline to Vertex AI

### ✏️ Items to configure before running
- Review / edit `config.yaml` for your project, bucket, and SQL queries
- Set `SELECTED_FEATURES_PATH` in Step 3 if your features file lives in GCS
- Confirm the `docker_uri` in `config.yaml` matches your training environment

---


## 0 · Imports


In [1]:
import os
import sys
import json
import datetime
import yaml

from pathlib import Path

from google.cloud import aiplatform

import kfp
from kfp import dsl
from kfp.compiler import Compiler

# ── Add the ss-models directory to sys.path so local helpers are importable ──
SS_MODELS_DIR = Path(os.getcwd())          # adjust if your cwd differs
if str(SS_MODELS_DIR) not in sys.path:
    sys.path.insert(0, str(SS_MODELS_DIR))

from pipeline_helpers import (
    build_and_upload_package,
    create_model_training_job_component,
)


## 1 · Load Configuration

All pipeline settings live in `config.yaml`.  
The cell below reads the file and prints key values so you can verify before submitting.


In [2]:
# ── Load config.yaml ─────────────────────────────────────────────────────────
CONFIG_PATH = SS_MODELS_DIR / 'config.yaml'

with open(CONFIG_PATH, 'r') as _f:
    config = yaml.safe_load(_f)

print('✅ Loaded config from:', CONFIG_PATH)
print()

# ── Convenience shortcuts ─────────────────────────────────────────────────────
gcp       = config['gcp']
vai       = config['vertex_ai']
bq_labels = config['bigquery_labels']
sql       = config['sql_queries']
data_proc = config['data_processing']
cat_cfg   = config['categorical']
feat_sel  = config['selected_features']
mdl       = config['model']
metrics   = config['metrics']
output    = config['output']
bq_qcfg   = config['bigquery_query']
mdl_reg   = config.get('model_registry', {})

PIPELINE_ROOT = vai['pipeline_root']

print('GCP project         :', gcp['project'])
print('BQ project          :', gcp['gcp_project'])
print('BQ dataset          :', gcp['gcp_db'])
print('Vertex AI location  :', vai['location'])
print('GCS pipeline root   :', PIPELINE_ROOT)
print('Docker URI          :', vai['docker_uri'])
print('Machine type        :', vai['machine_type'])
print('Outcome variable    :', data_proc['outcome_var'])
print('Num boost rounds    :', mdl['num_boost_round'])


✅ Loaded config from: c:\Users\A974930\Downloads\Github Action Test\vertex-pipeline-demo\src\ss-models\config.yaml

GCP project         : anbc-dev-hcm-cm-de
BQ project          : anbc-hcb-dev
BQ dataset          : clin_analytics_hcb_dev
Vertex AI location  : us-east4
GCS pipeline root   : gs://hcm-cm-de-code-hcb-dev/vertex-test/ss-models
Docker URI          : us-docker.pkg.dev/vertex-ai/training/xgboost-cpu.2-1:latest
Machine type        : n1-standard-16
Outcome variable    : cancer_case
Num boost rounds    : 3101


In [3]:
# ── CONSTANTS dict consumed by pipeline_helpers ───────────────────────────────
CONSTANTS = {
    'PROJECT':         gcp['project'],
    'COMPUTE_PROJECT': gcp['project'],
    'SHARED_PROJECT':  gcp['gcp_project'],
    'LOB':             bq_labels['lob'],
    'LOCATION':        vai['location'],
    'DOCKER_URI':      vai['docker_uri'],
    'SERVICE_ACCOUNT': vai['service_account'],
    'CMEK_KEY':        vai['cmek_key'],
    'LABELS': {
        'owner':         bq_labels['owner'],
        'costcenter':    str(bq_labels['costcenter']),
        'unique_id':     bq_labels['unique_id'],
        'lob':           bq_labels['lob'],
        'pipeline_type': bq_labels['pipeline_type'],
    },
}

# ── Machine spec for the training worker ─────────────────────────────────────
MACHINE_SPEC = {'machineType': vai['machine_type']}
if vai.get('accelerator_type'):
    MACHINE_SPEC['acceleratorType']  = vai['accelerator_type']
    MACHINE_SPEC['acceleratorCount'] = str(vai.get('accelerator_count', 1))

print('CONSTANTS ready.')
print('Machine spec:', MACHINE_SPEC)


CONSTANTS ready.
Machine spec: {'machineType': 'n1-standard-16'}


## 2 · Build & Upload the Training Package

Runs `python setup.py sdist` inside `SS_MODELS_DIR` and uploads the resulting  
`.tar.gz` to the GCS pipeline root.


In [4]:
DIST_DIR = SS_MODELS_DIR / 'dist'

PACKAGE_URI, _pkg_path = build_and_upload_package(
    training_dir  = str(SS_MODELS_DIR),
    output_dir    = str(DIST_DIR),
    project       = gcp['project'],
    pipeline_root = PIPELINE_ROOT,
)

print()
print('📦 Package URI:', PACKAGE_URI)


Building package from C:\Users\A974930\Downloads\Github Action Test\vertex-pipeline-demo\src\ss-models ...
STDOUT: running sdist
running egg_info
writing ss_models_trainer.egg-info\PKG-INFO
writing dependency_links to ss_models_trainer.egg-info\dependency_links.txt
writing entry points to ss_models_trainer.egg-info\entry_points.txt
writing requirements to ss_models_trainer.egg-info\requires.txt
writing top-level names to ss_models_trainer.egg-info\top_level.txt
reading manifest file 'ss_models_trainer.egg-info\SOURCES.txt'
writing manifest file 'ss_models_trainer.egg-info\SOURCES.txt'
running check
creating ss_models_trainer-0.1.0
creating ss_models_trainer-0.1.0\ss_models_trainer.egg-info
creating ss_models_trainer-0.1.0\trainer
creating ss_models_trainer-0.1.0\utils
copying files to ss_models_trainer-0.1.0...
copying setup.py -> ss_models_trainer-0.1.0
copying ss_models_trainer.egg-info\PKG-INFO -> ss_models_trainer-0.1.0\ss_models_trainer.egg-info
copying ss_models_trainer.egg-info\

## 3 · Assemble Training Arguments

The flat `[flag, value, ...]` list below is passed to `trainer.task` inside the container.  
Complex values (SQL dicts, hyperparameter dicts) are JSON-serialised here and then  
**automatically base64-encoded** by `pipeline_helpers` to avoid payload escaping issues.


In [5]:
# ── ✏️  Pre-selected features ─────────────────────────────────────────────────
# Edit SELECTED_FEATURES_LIST below to change which features are used in the model.
# To load from a GCS file instead, swap --selected-features-list for:
#   '--selected-features-path', 'gs://<bucket>/<path>/features.txt'
# For GCS, set e.g.:
#   SELECTED_FEATURES_PATH = 'gs://hcm-cm-de-code-hcb-dev/vertex-test/ss-models/xgboost_selected_features_cancer_v_case.txt'
SELECTED_FEATURES_LIST = [
    'emb131', 'emb169', 'emb186', 'uniq_dx_cd_cnt_3mo', 'emb14', 'change_q1_minus_q_2',
    'Oncologic_case_pre_12', 'other_spec_visits_pre_1', 'cost_9_12', 'pcp_visits_pre_9',
    'spec_visits_pre_6', 'emb194', 'emb116', 'lab_max_psa_1yr', 'emb94', 'emb238',
    'emb125', 'emb190', 'dx_cd_cnt_6mo', 'Endocrine_Metabolic_case_pre_12', 'emb42',
    'emb166', 'emb89', 'cancer_cv', 'Cardiac_case_pre_12', 'spec_visit_cnt_6mo', 'emb248',
    'cancer_visits_pre_9', 'prostate_cancer', 'emb145', 'emb152', 'Respiratory_case_pre_12',
    'emb250', 'cost_6_9', 'emb176', 'emb222', 'endo_cv', 'cancer_visits_pre_12', 'emb61',
    'emb6', 'emb80', 'emb74', 'emb118', 'emb233', 'emb46', 'spec_visits_pre_1', 'emb92',
    'emb57', 'emb66', 'emb197', 'emb244', 'pcp_visits_pre_1', 'emb91', 'colorectal_cancer',
    'lab_max_psa_2yr', 'emb170', 'lab_visit_cnt_2yr', 'Musculoskeletal_case_pre_12',
    'spec_visits_pre_12', 'emb185', 'emb31', 'age', 'emb225', 'emb202',
    'change_q1_minus_q_3', 'Digestive_case_pre_12', 'emb86', 'change_h_1_minus_h_2',
    'emb149', 'lab_visit_cnt_3mo', 'spec_visits_pre_9', 'emb29', 'card_visits_pre_12',
    'leukemia_myeloma', 'emb110', 'health_habits', 'emb35', 'breast_cancer', 'emb219',
    'cancer_visits_pre_3', 'emb242', 'emb224', 'cancer', 'dx_diff', 'Oncologic_case_pre_3',
    'emb21', 'cost_0_3', 'pcp_cv', 'emb187', 'emb73', 'emb175', 'social_risk_score',
    'emb142', 'emb2', 'brain_cancer', 'uniq_proc_cd_cnt_6mo', 'emb62', 'clm_ln_cnt_2yr',
    'emb160', 'Obstetric_case_pre_12', 'emb24', 'emb37', 'emb68', 'language_score',
    'uniq_rev_cd_cnt_3mo', 'emb77', 'cost_3_6', 'emb128', 'emb25', 'cancer_visits_pre_1',
    'clm_ln_cnt_1yr', 'emb214', 'er_clm_cnt_6mo', 'lab_visit_cnt_6mo', 'emb41', 'emb75',
    'emb137', 'emb234', 'emb85', 'emb56', 'emb207', 'Injury_Poisoning_case_pre_12',
    'emb203', 'rx_diff', 'emb32', 'emb182', 'emb232', 'uniq_proc_cd_cnt_3mo', 'emb228',
    'Renal_case_pre_12', 'Neurologic_case_pre_12', 'emb33', 'index_month', 'emb71',
    'hpd_count', 'emb143', 'emb235', 'emb159', 'emb247', 'emb120', 'emb236', 'other_cancer',
    'Infectious_case_pre_12', 'emb45', 'emb64', 'emb181', 'emb180', 'emb124',
    'pcp_visits_pre_6', 'Oncologic_case_pre_6', 'emb114', 'emb220', 'emb227',
    'ortho_visits_pre_3', 'change_q1_minus_q_4', 'emb147',
]

# ── Flat args list ────────────────────────────────────────────────────────────
TRAINING_ARGS = [
    # ── GCP / environment ────────────────────────────────────────────────────
    '--gcp-project',            gcp['project'],
    '--gcp-bq-project',         gcp['gcp_project'],
    '--gcp-db',                 gcp['gcp_db'],
    '--location',               gcp['location'],
    '--bucket-name',            gcp['bucket_name'],
    '--gcs-destination-path',   gcp['gcs_destination_path'],

    # ── BigQuery job labels ───────────────────────────────────────────────────
    '--owner',                  bq_labels['owner'],
    '--costcenter',             str(bq_labels['costcenter']),
    '--unique-id',              bq_labels['unique_id'],
    '--lob',                    bq_labels['lob'],
    '--pipeline-type',          bq_labels['pipeline_type'],
    '--expiration-days',        str(bq_labels['expiration_days']),

    # ── SQL queries (JSON-serialised; base64-encoded by helper) ───────────────
    '--sql-queries',            json.dumps(sql),

    # ── BigQuery read settings ────────────────────────────────────────────────
    '--bigquery-query-config',  json.dumps(bq_qcfg),

    # ── Data processing ───────────────────────────────────────────────────────
    '--random-seed',            str(data_proc['random_seed']),
    '--outcome-var',            data_proc['outcome_var'],
    '--indexing-var',           data_proc['indexing_var'],
    '--embedding-pattern',      data_proc['embedding_pattern'],
    '--test-size',              str(data_proc['test_size']),
    '--val-size',               str(data_proc['val_size']),

    # ── Categorical encoding map (JSON-serialised) ───────────────────────────
    '--categorical-config',     json.dumps(cat_cfg),

    # ── Pre-selected feature list (JSON-serialised; base64-encoded by helper) ─
    '--selected-features-list', json.dumps(SELECTED_FEATURES_LIST),

    # ── XGBoost hyperparameters (JSON-serialised) ────────────────────────────
    '--model-params',           json.dumps(mdl['params']),
    '--num-boost-round',        str(mdl['num_boost_round']),
    '--verbose-eval',           str(mdl['verbose_eval']),

    # ── Evaluation percentiles ────────────────────────────────────────────────
    '--percentiles',            json.dumps(metrics['percentiles']),

    # ── Output settings (JSON-serialised) ────────────────────────────────────
    '--output-config',          json.dumps(output),

    # ── Model identity / governance labels ───────────────────────────────────
    # These are attached to the Vertex AI Model Registry entry.
    # Empty values are silently ignored by task.py.
    '--model-name',                       config.get('model_labels', {}).get('model_name', ''),
    '--tenant',                           config.get('model_labels', {}).get('tenant', ''),
    '--self-serve',                       config.get('model_labels', {}).get('self_serve', 'true'),
    '--vertex-model-id',                  config.get('model_labels', {}).get('vertex_model_id', 'none'),
    '--vertex-model-version-alias',       config.get('model_labels', {}).get('vertex_model_version_alias', 'none'),

    # ── Vertex AI Model Registry ──────────────────────────────────────────────
    # Model is always saved as model.bst and uploaded to a dedicated GCS folder.
    # Set model_registry.display_name in config.yaml to enable auto-registration.
    '--model-registry-display-name',      mdl_reg.get('display_name', ''),
    '--serving-container-image-uri',      mdl_reg.get('serving_container_image_uri', ''),
    '--model-registry-location',          vai['location'],
    '--model-registry-cmek-key',          vai['cmek_key'],
    '--model-registry-service-account',   vai['service_account'],
    '--upload-to-existing-model',         str(mdl_reg.get('upload_to_existing_model', False)).lower(),
    '--existing-model-resource-name',     mdl_reg.get('existing_model_resource_name', ''),
    '--model-description',                mdl_reg.get('description', ''),

    # ── Batch prediction (optional) ───────────────────────────────────────────
    # Overrides the batch_prediction section in config.yaml when set.
    # Set enable=true and fill input_table / output_table to activate.
    '--batch-predict-config',             json.dumps(config.get('batch_prediction', {})),
]

bp_cfg = config.get('batch_prediction', {})
print(f'✅ {len(TRAINING_ARGS) // 2} argument pairs assembled.')
print(f'   Selected features    : {len(SELECTED_FEATURES_LIST)} features inline')
print(f'   Model Registry name  : {mdl_reg.get("display_name") or "(skipped – set display_name to enable)"}')
print(f'   Batch prediction     : {"enabled  → " + bp_cfg.get("output_table","") if bp_cfg.get("enable") else "disabled (set batch_prediction.enable: true to run)"}')


✅ 41 argument pairs assembled.
   Selected features    : 156 features inline
   Model Registry name  : ss-xgboost-model
   Batch prediction     : enabled  → anbc-hcb-dev.clin_analytics_hcb_dev.ss_model_test_predictions


## 4 · Define the KFP Pipeline

A single-step pipeline that launches the model training job.


In [6]:
# ── Training-container env vars (EXTRA_ENV) ───────────────────────────────────
# These are the ONLY variables injected into the Vertex AI training container.
#
# NOTE: config.yaml → environment_variables is NOT sourced here.
#       Those values are exclusively for the Vertex AI Model Registry
#       (serving_container_environment_variables) and are attached to the
#       model by model_card.py after training completes.
#
# Add here only values the training job itself needs at runtime.

import time as _time

EXTRA_ENV = {
    # ── Feature list (JSON array) — used by task.py and model_card.py ─────────
    # model_card._build_serving_env() also reads this from os.environ to attach
    # it to the registered model's serving container env vars.
    'FEATURE_NAMES':   json.dumps(SELECTED_FEATURES_LIST),

    # ── Unique timestamp for this training run ────────────────────────────────
    'MODEL_TIMESTAMP': str(int(_time.time())),
}

print(f'✅ EXTRA_ENV assembled — {len(EXTRA_ENV)} variable(s) will be injected into the training container.')
print(f'   FEATURE_NAMES  : {len(SELECTED_FEATURES_LIST)} features')
print(f'   MODEL_TIMESTAMP: {EXTRA_ENV["MODEL_TIMESTAMP"]}')
print()
print('ℹ️  All other env vars (CMEK keys, paths, model settings, etc.) are attached')
print('   directly to the Model Registry entry by model_card._build_serving_env().')

@dsl.pipeline(
    name='model-training-pipeline',
    description='XGBoost model training using pre-selected features (ss-models)',
)
def model_training_pipeline():
    """Single-step KFP pipeline: load data → feature encode → train → save → upload predictions."""

    _training_op = create_model_training_job_component(
        pipeline_root  = PIPELINE_ROOT,
        constants      = CONSTANTS,
        machine_type   = MACHINE_SPEC,
        package_uris   = [PACKAGE_URI],
        python_module  = 'trainer.task',
        display_name   = 'xgb-model-training',
        env            = EXTRA_ENV,
        args           = TRAINING_ARGS,
    )

print('✅ Pipeline function defined.')


✅ EXTRA_ENV assembled — 2 variable(s) will be injected into the training container.
   FEATURE_NAMES  : 156 features
   MODEL_TIMESTAMP: 1771948645

ℹ️  All other env vars (CMEK keys, paths, model settings, etc.) are attached
   directly to the Model Registry entry by model_card._build_serving_env().
✅ Pipeline function defined.


## 5 · Compile the Pipeline to JSON


In [7]:
PIPELINE_JSON = str(SS_MODELS_DIR / 'model_training_pipeline.json')

Compiler().compile(
    pipeline_func = model_training_pipeline,
    package_path  = PIPELINE_JSON,
)

print('✅ Pipeline compiled to:', PIPELINE_JSON)


✅ Pipeline compiled to: c:\Users\A974930\Downloads\Github Action Test\vertex-pipeline-demo\src\ss-models\model_training_pipeline.json


## 6 · Submit the Pipeline to Vertex AI

> **Tip:** Run all cells above first to verify the config and package build before hitting submit.


In [8]:
# ── Initialise the Vertex AI SDK ──────────────────────────────────────────────
aiplatform.init(
    project        = gcp['project'],
    location       = vai['location'],
    staging_bucket = f"gs://{gcp['bucket_name']}",
)

print('✅ Vertex AI SDK initialised.')

print('   Project  :', gcp['project'])
print('   Location :', vai['location'])


✅ Vertex AI SDK initialised.
   Project  : anbc-dev-hcm-cm-de
   Location : us-east4


In [9]:
# ── Unique run ID ─────────────────────────────────────────────────────────────
RUN_TIMESTAMP   = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
PIPELINE_JOB_ID = f'model-training-{RUN_TIMESTAMP}'

# ── Create and submit the PipelineJob ─────────────────────────────────────────
pipeline_job = aiplatform.PipelineJob(
    display_name              = PIPELINE_JOB_ID,
    template_path             = PIPELINE_JSON,
    pipeline_root             = PIPELINE_ROOT,
    project                   = gcp['project'],
    location                  = vai['location'],
    enable_caching            = False,
    encryption_spec_key_name  = CONSTANTS['CMEK_KEY'],
    labels                    = CONSTANTS['LABELS'],
)

pipeline_job.submit(
    service_account = vai['service_account']
)

print()
print('🚀 Pipeline submitted!')
print('   Job ID        :', PIPELINE_JOB_ID)
print('   Pipeline root :', PIPELINE_ROOT)
print('   Console       → Vertex AI > Pipelines > Pipeline Runs')


Creating PipelineJob
PipelineJob created. Resource name: projects/46378383599/locations/us-east4/pipelineJobs/model-training-pipeline-20260224105730
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/46378383599/locations/us-east4/pipelineJobs/model-training-pipeline-20260224105730')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-east4/pipelines/runs/model-training-pipeline-20260224105730?project=46378383599

🚀 Pipeline submitted!
   Job ID        : model-training-20260224-105725
   Pipeline root : gs://hcm-cm-de-code-hcb-dev/vertex-test/ss-models
   Console       → Vertex AI > Pipelines > Pipeline Runs


In [33]:
# ── Optional: block notebook execution until the pipeline finishes ────────────
# Comment this out if you want submit-and-forget behaviour.

pipeline_job.wait()

print()
print('Pipeline final status:', pipeline_job.state)


PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/cancer-model-training-pipeline-20260222145924 current state:
3
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/cancer-model-training-pipeline-20260222145924 current state:
3
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/cancer-model-training-pipeline-20260222145924 current state:
3
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/cancer-model-training-pipeline-20260222145924 current state:
3
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/cancer-model-training-pipeline-20260222145924 current state:
3
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/cancer-model-training-pipeline-20260222145924 current state:
3
PipelineJob run completed. Resource name: projects/46378383599/locations/us-east4/pipelineJobs/cancer-model-training-pipeline-20260222145924

Pipeline final status: 4
